# Data Import and Intial Overview

In [ ]:
# Dependencies are installed in the selected project .venv.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Raw").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the project Data/Raw directory from this notebook's working directory.")
RAW_DIR = ROOT / "Data" / "Raw"
PROCESSED_DIR = ROOT / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PAIR_KEY = ["problem_id", "submission_id_v0", "submission_id_v1"]
REQUIRED_COLUMNS = [
    "user_id", "problem_id", "language", "submission_id_v0", "submission_id_v1",
    "code_v0_no_empty_lines", "code_v1_no_empty_lines", "result_energy_v0",
    "result_time_v0", "result_memory_v0", "result_energy_v1", "result_time_v1",
    "result_memory_v1",
]
RESOURCE_COLUMNS = [
    "result_energy_v0", "result_time_v0", "result_memory_v0",
    "result_energy_v1", "result_time_v1", "result_memory_v1",
]
RAW_FILES = {
    "validated": "public_validated_energy_results.jsonl",
    "significant": "public_significant_energy_diff.jsonl",
    "classified": "public_classified_smells.jsonl",
}

def load_and_clean_dataset(name, filename):
    path = RAW_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Required raw dataset is missing: {path}")
    frame = pd.read_json(path, lines=True)
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{filename} is missing required columns: {missing_columns}")

    before_rows = len(frame)
    missing_required = int(frame[REQUIRED_COLUMNS].isna().any(axis=1).sum())
    frame = frame.dropna(subset=REQUIRED_COLUMNS).copy()
    for column in RESOURCE_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    valid_metrics = np.isfinite(frame[RESOURCE_COLUMNS]).all(axis=1)
    valid_metrics &= frame[RESOURCE_COLUMNS].ge(0).all(axis=1)
    valid_code = frame["code_v0_no_empty_lines"].astype(str).str.strip().ne("")
    valid_code &= frame["code_v1_no_empty_lines"].astype(str).str.strip().ne("")
    invalid_measurements_or_code = int((~(valid_metrics & valid_code)).sum())
    frame = frame.loc[valid_metrics & valid_code].copy()

    duplicate_pairs = int(frame.duplicated(subset=PAIR_KEY, keep="first").sum())
    frame = frame.drop_duplicates(subset=PAIR_KEY, keep="first").reset_index(drop=True)

    # Optional annotation text is retained; nested values are serialized losslessly for Parquet.
    for column in frame.columns:
        if column in REQUIRED_COLUMNS or column in RESOURCE_COLUMNS:
            continue
        if frame[column].dtype == "object":
            frame[column] = frame[column].map(
                lambda value: "" if value is None or (isinstance(value, float) and np.isnan(value)) else value
            )
            if frame[column].map(lambda value: isinstance(value, (dict, list))).any():
                frame[column] = frame[column].map(
                    lambda value: json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
                )

    report = {
        "dataset": name,
        "raw_rows": before_rows,
        "rows_with_missing_required_values_removed": missing_required,
        "rows_with_invalid_metrics_or_empty_code_removed": invalid_measurements_or_code,
        "duplicate_pairs_removed": duplicate_pairs,
        "clean_rows": len(frame),
    }
    return frame, report

# Load each original source independently. Raw files remain unchanged.
df_full, validated_quality = load_and_clean_dataset("validated", RAW_FILES["validated"])
df_significant, significant_quality = load_and_clean_dataset("significant", RAW_FILES["significant"])
df_classified, classified_quality = load_and_clean_dataset("classified", RAW_FILES["classified"])
quality_report = pd.DataFrame([validated_quality, significant_quality, classified_quality])
print(quality_report.to_string(index=False))


In [ ]:
df_full.info()

In [ ]:
# Display the first 5 rows of the dataset
print("\nFirst 5 Rows of the Dataset:")
display(df_full.head())

In [ ]:
# Display the number of records and columns
print(f"\nNumber of records: {df_full.shape[0]}")
print(f"Number of columns: {df_full.shape[1]}")

### Data Dictoniary Table

| Column | Description | Data Type | Role | Summary / Range |
|---|---|---|---|---|
| `problem_id` | Programming task identifier | Categorical (object) | Grouping | 1,232 unique programming tasks |
| `code_v0_no_empty_lines` | Source code of implementation 0 | Text (object) | Input for feature extraction | 21,428 valid implementations |
| `code_v1_no_empty_lines` | Source code of implementation 1 | Text (object) | Input for feature extraction | 21,428 valid implementations |
| `result_energy_v0` | Energy consumption of implementation 0 | Numeric (float64) | Measurement | [352.01, 35307.72] Joules |
| `result_energy_v1` | Energy consumption of implementation 1 | Numeric (float64) | Measurement | [352.01, 35467.43] Joules |
| `result_time_v0` | Execution time of implementation 0 | Numeric (float64) | Measurement | [0.03, 110.87] seconds |
| `result_time_v1` | Execution time of implementation 1 | Numeric (float64) | Measurement | [0.03, 110.37] seconds |
| `result_memory_v0` | Memory usage of implementation 0 | Numeric (int64) | Measurement | [3072, 280388] KB |
| `result_memory_v1` | Memory usage of implementation 1 | Numeric (int64) | Measurement | [3072, 281864] KB |

# Data Quality Assements

In [ ]:
# Data Quality Assessment: Check for Missing Values
print("\n--- Missing Values Assessment ---")
missing_values = df_full.isnull().sum()
missing_percentage = (df_full.isnull().sum() / len(df_full)) * 100

missing_info = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

# Filter to show only columns with missing values
missing_info = missing_info[missing_info['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)

if missing_info.empty:
    print("No missing values found in the dataset.")
else:
    print("Missing Values per Column:")
    display(missing_info)

In [ ]:
# Data Quality Assessment: Check for Duplicate Records
print("\n--- Duplicate Records Assessment ---")
# A record is considered a duplicate if all its columns are identical.
duplicate_rows = df_full[df_full.duplicated()]

# Also check for 'logical duplicates' based on identifying columns
# What defines a unique experiment or comparison?
# A pair of submissions for a problem_id should be unique.
# The combination of problem_id, submission_id_v0, and submission_id_v1 should ideally be unique.
logical_duplicate_cols = ['problem_id', 'submission_id_v0', 'submission_id_v1']
duplicate_logical_entries = df_full[df_full.duplicated(subset=logical_duplicate_cols, keep=False)]

print(f"Total fully duplicate rows: {len(duplicate_rows)}")
if not duplicate_rows.empty:
    print("Sample fully duplicate rows:")
    display(duplicate_rows.head())

print(f"\nTotal logical duplicate entries (based on problem_id, submission_id_v0, submission_id_v1): {len(duplicate_logical_entries)}")
if not duplicate_logical_entries.empty:
    print("Sample logical duplicate entries:")
    display(duplicate_logical_entries.sort_values(by=logical_duplicate_cols).head())

In [ ]:
# Check for Invalid, Inconsistent, or Unexpected Values
print("\n--- Invalid/Inconsistent Values Assessment ---")

# Check 'language' column
print("\nUnique values in 'language' column:")
print(df_full['language'].value_counts())

# Check descriptive statistics for numerical resource columns
resource_cols = [
    'result_energy_v0', 'result_time_v0', 'result_memory_v0',
    'result_energy_v1', 'result_time_v1', 'result_memory_v1'
]

print("\nDescriptive Statistics for Resource Metrics:")
display(df_full[resource_cols].describe())

In [ ]:
# Outlier Detection for Numerical Measurements
print("\n--- Outlier Detection ---")

# Using IQR method to identify potential outliers
for col in resource_cols:
    Q1 = df_full[col].quantile(0.25)
    Q3 = df_full[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df_full[(df_full[col] < lower_bound) | (df_full[col] > upper_bound)]
    print(f"\nPotential outliers in '{col}': {len(outliers)} records ({len(outliers)/len(df_full)*100:.2f}%) ")
    if not outliers.empty:
        # Display some sample outliers to understand their nature
        print("Sample Outliers (top 5 by value):")
        display(outliers.sort_values(by=col, ascending=False).head())


### Data Validation: Checking for Negative, Zero, and Malformed Measurements

In [ ]:
import pandas as pd

# 1. Check for negative measurements
print("=== 1. Checking for Negative Values ===")
neg_energy_v0 = (df_full['result_energy_v0'] < 0).sum()
neg_energy_v1 = (df_full['result_energy_v1'] < 0).sum()
neg_time_v0 = (df_full['result_time_v0'] < 0).sum()
neg_time_v1 = (df_full['result_time_v1'] < 0).sum()
neg_mem_v0 = (df_full['result_memory_v0'] < 0).sum()
neg_mem_v1 = (df_full['result_memory_v1'] < 0).sum()

print(f"Negative Energy (v0): {neg_energy_v0} | (v1): {neg_energy_v1}")
print(f"Negative Time (v0): {neg_time_v0} | (v1): {neg_time_v1}")
print(f"Negative Memory (v0): {neg_mem_v0} | (v1): {neg_mem_v1}")

# 2. Check for zero measurements
print("\n=== 2. Checking for Zero Values ===")
zero_energy_v0 = (df_full['result_energy_v0'] == 0).sum()
zero_energy_v1 = (df_full['result_energy_v1'] == 0).sum()
zero_time_v0 = (df_full['result_time_v0'] == 0).sum()
zero_time_v1 = (df_full['result_time_v1'] == 0).sum()
zero_mem_v0 = (df_full['result_memory_v0'] == 0).sum()
zero_mem_v1 = (df_full['result_memory_v1'] == 0).sum()

print(f"Zero Energy (v0): {zero_energy_v0} | (v1): {zero_energy_v1}")
print(f"Zero Time (v0): {zero_time_v0} | (v1): {zero_time_v1}")
print(f"Zero Memory (v0): {zero_mem_v0} | (v1): {zero_mem_v1}")

# 3. Check for invalid or empty source code
print("\n=== 3. Checking for Empty/Malformed Code ===")
empty_code_v0 = df_full['code_v0_no_empty_lines'].apply(lambda x: str(x).strip() == '').sum()
empty_code_v1 = df_full['code_v1_no_empty_lines'].apply(lambda x: str(x).strip() == '').sum()
print(f"Empty/Whitespace-only code (v0): {empty_code_v0}")
print(f"Empty/Whitespace-only code (v1): {empty_code_v1}")

# 4. Check for incomplete or unmatched pairs
print("\n=== 4. Checking for Unmatched Pairs ===")
missing_sub_v0 = df_full['submission_id_v0'].isnull().sum()
missing_sub_v1 = df_full['submission_id_v1'].isnull().sum()
print(f"Missing submission identifier (v0): {missing_sub_v0}")
print(f"Missing submission identifier (v1): {missing_sub_v1}")

### Check for Non-ASCII or Unexpected Characters in Source Code

In [ ]:
def check_non_ascii(code_str):
    if not isinstance(code_str, str):
        return []
    # Report characters outside printable ASCII while keeping the diagnostic Windows-console safe.
    return [char for char in code_str if ord(char) > 127 or (ord(char) < 32 and char not in "\n\r\t")]

df_full["non_ascii_v0"] = df_full["code_v0_no_empty_lines"].apply(check_non_ascii)
df_full["non_ascii_v1"] = df_full["code_v1_no_empty_lines"].apply(check_non_ascii)
rows_with_non_ascii_v0 = df_full[df_full["non_ascii_v0"].map(len) > 0]
rows_with_non_ascii_v1 = df_full[df_full["non_ascii_v1"].map(len) > 0]
print(f"Rows with unusual characters in v0 code: {len(rows_with_non_ascii_v0)}")
print(f"Rows with unusual characters in v1 code: {len(rows_with_non_ascii_v1)}")

if len(rows_with_non_ascii_v0) > 0 or len(rows_with_non_ascii_v1) > 0:
    print("\n--- Sample records with unusual characters ---")
    unique_chars = set()
    for chars in df_full["non_ascii_v0"]:
        unique_chars.update(chars)
    for chars in df_full["non_ascii_v1"]:
        unique_chars.update(chars)
    print(f"Unique non-standard characters found: {ascii(list(unique_chars)[:50])}")
    samples = df_full[
        (df_full["non_ascii_v0"].map(len) > 0) | (df_full["non_ascii_v1"].map(len) > 0)
    ].head(5)
    samples["code_v0_no_empty_lines"] = samples["code_v0_no_empty_lines"].map(lambda code: ascii(code[:300]))
    samples["code_v1_no_empty_lines"] = samples["code_v1_no_empty_lines"].map(lambda code: ascii(code[:300]))
    samples["non_ascii_v0"] = samples["non_ascii_v0"].map(ascii)
    samples["non_ascii_v1"] = samples["non_ascii_v1"].map(ascii)
    display(samples[[
        "problem_id", "code_v0_no_empty_lines", "non_ascii_v0",
        "code_v1_no_empty_lines", "non_ascii_v1",
    ]])
df_full = df_full.drop(columns=["non_ascii_v0", "non_ascii_v1"])


### Identify algorithms and comparable implementations

In [ ]:
# Identify all algorithms, programming tasks, or implementation categories
print("\n--- Algorithm/Task Identification ---")

# Group by 'problem_id' to identify unique tasks using df_full
unique_problems = df_full['problem_id'].nunique()
print(f"Number of unique programming problems (tasks): {unique_problems}")

# Count records per problem_id
records_per_problem = df_full['problem_id'].value_counts()
print("\nTop 10 problems by number of records:")
display(records_per_problem.head(10))

print("\nBottom 10 problems by number of records:")
display(records_per_problem.tail(10))

In [ ]:
# Determine whether the dataset contains multiple implementations of the same task.
print("\n--- Multiple Implementations per Task ---")

# Extract all submission IDs for v0 and v1
all_submission_ids = pd.concat([
    df_full[['problem_id', 'submission_id_v0']].rename(columns={'submission_id_v0': 'submission_id'}),
    df_full[['problem_id', 'submission_id_v1']].rename(columns={'submission_id_v1': 'submission_id'})
])

# Count distinct submissions per problem_id
distinct_implementations_per_problem = all_submission_ids.groupby('problem_id')['submission_id'].nunique()

print("Distribution of distinct implementations per problem_id:")
print(distinct_implementations_per_problem.value_counts().sort_index())

# Identify problems with at least 3 distinct implementations
problems_with_3_plus_impl = distinct_implementations_per_problem[distinct_implementations_per_problem >= 3]

print(f"\nNumber of problems with at least 3 distinct implementations: {len(problems_with_3_plus_impl)}")
if not problems_with_3_plus_impl.empty:
    print("Sample problems with 3+ distinct implementations (problem_id and count):")
    display(problems_with_3_plus_impl.head())

In [ ]:
# How many different problems?
print("Unique problems:", df_full["problem_id"].nunique())

# How many records per problem?
print(df_full["problem_id"].value_counts().head(20))

# What classifications exist?
if "final_classification" in df_full.columns:
    print(df_full["final_classification"].value_counts())
elif 'df_sig' in globals() and "final_classification" in df_sig.columns:
    print(df_sig["final_classification"].value_counts())
else:
    print("Column 'final_classification' not found in current main DataFrame.")

# Display one example pair of code submissions
row = df_full.iloc[0]

print("Problem ID:", row["problem_id"])
print("Language:", row["language"])

print("\nVersion 0:\n", row["code_v0_no_empty_lines"])
print("\nVersion 1:\n", row["code_v1_no_empty_lines"])

print("\nEnergy v0:", row["result_energy_v0"])
print("Energy v1:", row["result_energy_v1"])

# Mapping Problem ID to Problem names
To better understand the dataset

In [ ]:
import pandas as pd

# The raw files have no task-title column; use observed problem IDs without a network dependency.
problems = pd.DataFrame({"id": sorted(df_full["problem_id"].dropna().unique())})
problems["slug_name"] = problems["id"]
print(f"Using {len(problems)} observed problem IDs; no external metadata download is required.")


In [ ]:
print(problems.columns.tolist())

In [ ]:
print(problems.head(10).to_string())

In [ ]:
print("\n--- Merging problem names ---")
if 'problems' in globals() and not problems.empty:
    df_full = pd.merge(df_full, problems[['id', 'slug_name']], left_on='problem_id', right_on='id', how='left')
    df_full = df_full.rename(columns={'slug_name': 'problem_name'}).drop(columns=['id'])
else:
    print("HuggingFace dataset not loaded. Creating placeholder problem names.")
    df_full['problem_name'] = df_full['problem_id']

print("problem_name column processed. Displaying first few rows:")
display(df_full[['problem_id', 'problem_name']].head())

In [ ]:
# Verify that problem_name has been added to the DataFrame
print("\nUpdated DataFrame columns:")
print(df_full.columns.tolist())

In [ ]:
import pandas as pd

if 'problems' in globals() and not problems.empty:
    problem_table = problems[
        ["id", "slug_name"]
    ].drop_duplicates().rename(columns={"id": "problem_id", "slug_name": "problem_name"})
else:
    print("Using problem_id as fallback for problem_table.")
    problem_table = pd.DataFrame({
        'problem_id': df_full['problem_id'].unique(),
        'problem_name': df_full['problem_id'].unique()
    })

implementations_v0 = df_full[['problem_id', 'submission_id_v0', 'language']].rename(columns={'submission_id_v0': 'submission_id'})
implementations_v1 = df_full[['problem_id', 'submission_id_v1', 'language']].rename(columns={'submission_id_v1': 'submission_id'})
implementation_table = pd.concat([implementations_v0, implementations_v1]).drop_duplicates()

print("problem_table created successfully with columns:", problem_table.columns.tolist())
print("implementation_table created successfully with columns:", implementation_table.columns.tolist())

In [ ]:
# Persist cleaned copies; original JSONL files in Data/Raw are never modified.
processed_datasets = {
    "validated_energy_results": df_full,
    "significant_energy_diff": df_significant,
    "classified_smells": df_classified,
}
for name, frame in processed_datasets.items():
    output_path = PROCESSED_DIR / f"{name}.parquet"
    frame.to_parquet(output_path, index=False)
    print(f"Saved {len(frame):,} rows to {output_path.relative_to(ROOT)}")
quality_report.to_csv(PROCESSED_DIR / "data_quality_report.csv", index=False)
